# Corrective RAG (CRAG): Retrieval-Augmented Generation with Dynamic Correction

## Overview

Corrective RAG extends standard RAG by **evaluating** whether retrieved documents are actually useful, and **correcting** the approach when they're not:

| Relevance Score | Action | Source |
|---|---|---|
| **High** (> 0.7) | Use the retrieved document directly | Local knowledge base |
| **Low** (< 0.3) | Discard documents, do a **web search** instead | Web |
| **Ambiguous** (0.3–0.7) | Combine retrieved document **+** web search | Both |

This way, the system never generates from irrelevant context — it either trusts local documents or falls back to the web.

## Models Used

- **LLM**: `gemma3:12b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)
- **Web search**: DuckDuckGo (no API key needed)

<div style="text-align: center;">

<img src="./images/crag.svg" alt="Corrective RAG" style="width:80%; height:auto;">
</div>

---
## Step 0: Import Packages

In [1]:
from langchain_core.prompts import PromptTemplate
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.tools import DuckDuckGoSearchResults
import json

---
## Step 1: Set Up the LLM and Embedding Model

In [2]:
llm = ChatOllama(model="gemma3:12b", max_tokens=1000, temperature=0)
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load the PDF and Create a Vector Store

In [3]:
path = "data/Understanding_Climate_Change.pdf"

loader = PyPDFLoader(path)
documents = loader.load()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(documents)

vectorstore = FAISS.from_documents(splits, embedding_model)

print(f"Loaded {len(documents)} pages, split into {len(splits)} chunks")
print(f"Vector store created")

Loaded 33 pages, split into 97 chunks
Vector store created


---
## Step 3: Initialize the Web Search Tool

DuckDuckGo search is our fallback when the local knowledge base doesn't have relevant information.

In [4]:
search = DuckDuckGoSearchResults()

print("Web search tool ready")

Web search tool ready


---
## Step 4: Define the LLM Chains for Each CRAG Step

CRAG uses 3 specialized LLM chains:

1. **Retrieval Evaluator** — Scores how relevant a document is to the query (0 to 1).
2. **Knowledge Refiner** — Extracts key bullet points from a document.
3. **Query Rewriter** — Rewrites the query to be better suited for web search.

In [5]:
# --- 1. Retrieval Evaluator (scores 0 to 1) ---
evaluator_schema = {
    "title": "RetrievalEvaluator",
    "type": "object",
    "properties": {
        "relevance_score": {
            "type": "number",
            "description": "The relevance score of the document to the query, between 0 and 1"
        }
    },
    "required": ["relevance_score"]
}
evaluator_prompt = PromptTemplate(
    input_variables=["query", "document"],
    template="On a scale from 0 to 1, how relevant is the following document to the query? Query: {query}\nDocument: {document}\nRelevance score:"
)
evaluator_chain = evaluator_prompt | llm.with_structured_output(evaluator_schema)

# --- 2. Knowledge Refiner (extracts key points) ---
refiner_schema = {
    "title": "KnowledgeRefinement",
    "type": "object",
    "properties": {
        "key_points": {
            "type": "string",
            "description": "Key information extracted from the document in bullet points"
        }
    },
    "required": ["key_points"]
}
refiner_prompt = PromptTemplate(
    input_variables=["document"],
    template="Extract the key information from the following document in bullet points:\n{document}\nKey points:"
)
refiner_chain = refiner_prompt | llm.with_structured_output(refiner_schema)

# --- 3. Query Rewriter (for web search) ---
rewriter_schema = {
    "title": "QueryRewriter",
    "type": "object",
    "properties": {
        "query": {
            "type": "string",
            "description": "The rewritten query optimized for web search"
        }
    },
    "required": ["query"]
}
rewriter_prompt = PromptTemplate(
    input_variables=["query"],
    template="Rewrite the following query to make it more suitable for a web search:\n{query}\nRewritten query:"
)
rewriter_chain = rewriter_prompt | llm.with_structured_output(rewriter_schema)

# --- Response generation prompt (used at the end) ---
response_prompt = PromptTemplate(
    input_variables=["query", "knowledge", "sources"],
    template=(
        "Based on the following knowledge, answer the query. "
        "Include the sources with their links (if available) at the end of your answer:\n"
        "Query: {query}\nKnowledge: {knowledge}\nSources: {sources}\nAnswer:"
    )
)
response_chain = response_prompt | llm

print("All CRAG chains ready")

All CRAG chains ready


---
---
# Test 1: High-Relevance Query (Climate Change)

The query is about climate change — the document should be highly relevant.

Expected flow: Retrieve → Score high → Use retrieved document → Generate answer.

---
## Step 5: Retrieve Documents

In [6]:
query = "What are the main causes of climate change?"
print(f"Query: {query}\n")

top_k = 3
docs = vectorstore.similarity_search(query, k=top_k)
retrieved_docs = [doc.page_content for doc in docs]

print(f"Retrieved {len(retrieved_docs)} documents:")
for i, doc in enumerate(retrieved_docs, 1):
    print(f"\n  Doc {i}: {doc[:150]}...")

Query: What are the main causes of climate change?

Retrieved 3 documents:

  Doc 1: Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphe...

  Doc 2: Most of these climate changes are attributed to very small variations in Earth's orbit that 
change the amount of solar energy our planet receives. Du...

  Doc 3: Understanding Climate Change 
Chapter 1: Introduction to Climate Change 
Climate change refers to significant, long-term changes in the global climate...


---
## Step 6: Evaluate Relevance of Each Document

The LLM scores each document from 0 (irrelevant) to 1 (perfectly relevant).

In [7]:
eval_scores = []
for i, doc in enumerate(retrieved_docs, 1):
    score = evaluator_chain.invoke({"query": query, "document": doc})["relevance_score"]
    eval_scores.append(float(score))
    print(f"Doc {i}: relevance = {score:.2f}")

max_score = max(eval_scores)
print(f"\nHighest score: {max_score:.2f}")

Doc 1: relevance = 0.95
Doc 2: relevance = 0.85
Doc 3: relevance = 0.90

Highest score: 0.95


---
## Step 7: Decide Action Based on Relevance Score

| Score | Action |
|---|---|
| > 0.7 | **Correct** — use the retrieved document as-is |
| < 0.3 | **Incorrect** — discard documents, do a web search |
| 0.3–0.7 | **Ambiguous** — combine retrieved document + web search |

In [8]:
sources = []

if max_score > 0.7:
    print("Action: CORRECT — Using retrieved document")
    best_doc = retrieved_docs[eval_scores.index(max_score)]
    final_knowledge = best_doc
    sources.append(("Retrieved document", ""))

elif max_score < 0.3:
    print("Action: INCORRECT — Performing web search")

    # Rewrite query for web search
    rewritten_query = rewriter_chain.invoke({"query": query})["query"].strip()
    print(f"Rewritten query: {rewritten_query}")

    # Search the web
    web_results = search.run(rewritten_query)
    print(f"Web results received")

    # Refine web results into key points
    key_points_text = refiner_chain.invoke({"document": web_results})["key_points"]
    final_knowledge = key_points_text

    # Parse sources
    try:
        parsed = json.loads(web_results)
        sources = [(r.get("title", "Untitled"), r.get("link", "")) for r in parsed]
    except json.JSONDecodeError:
        sources = [("Web search", "")]

else:
    print("Action: AMBIGUOUS — Combining retrieved document + web search")
    best_doc = retrieved_docs[eval_scores.index(max_score)]

    # Refine the retrieved document
    retrieved_key_points = refiner_chain.invoke({"document": best_doc})["key_points"]

    # Also do a web search
    rewritten_query = rewriter_chain.invoke({"query": query})["query"].strip()
    web_results = search.run(rewritten_query)
    web_key_points = refiner_chain.invoke({"document": web_results})["key_points"]

    final_knowledge = retrieved_key_points + "\n" + web_key_points
    sources = [("Retrieved document", "")]
    try:
        parsed = json.loads(web_results)
        sources += [(r.get("title", "Untitled"), r.get("link", "")) for r in parsed]
    except json.JSONDecodeError:
        sources.append(("Web search", ""))

print(f"\nFinal knowledge preview: {str(final_knowledge)[:300]}...")
print(f"\nSources:")
for title, link in sources:
    print(f"  {title}: {link}" if link else f"  {title}")

Action: CORRECT — Using retrieved document

Final knowledge preview: Chapter 2: Causes of Climate Change 
Greenhouse Gases 
The primary cause of recent climate change is the increase in greenhouse gases in the 
atmosphere. Greenhouse gases, such as carbon dioxide (CO2), methane (CH4), and nitrous 
oxide (N2O), trap heat from the sun, creating a "greenhouse effect." T...

Sources:
  Retrieved document


---
## Step 8: Generate the Final Answer

In [9]:
sources_text = "\n".join([f"{title}: {link}" if link else title for title, link in sources])

answer = response_chain.invoke({
    "query": query,
    "knowledge": final_knowledge,
    "sources": sources_text
}).content

print(f"Question: {query}")
print(f"\nAnswer: {answer}")

Question: What are the main causes of climate change?

Answer: The main causes of climate change are the increase in greenhouse gases in the atmosphere, primarily due to human activities. These gases, including carbon dioxide (CO2), methane (CH4), and nitrous oxide (N2O), trap heat and intensify the "greenhouse effect." A significant contributor to this increase is the burning of fossil fuels (coal, oil, and natural gas) for energy, which began to rise significantly during the Industrial Revolution and continues today.



**Sources:**

*   Retrieved document (Chapter 2: Causes of Climate Change) - No direct link provided.


---
---
# Test 2: Irrelevant Query (Harry Potter)

The document is about climate change, but the query is about Harry Potter.

Expected flow: Retrieve → Score low → **Web search** → Generate answer from web results.

---
## Step 9: Run the Full CRAG Pipeline on an Irrelevant Query

In [10]:
query2 = "how did harry beat quirrell?"
print(f"Query: {query2}\n")

# --- Retrieve ---
docs2 = vectorstore.similarity_search(query2, k=3)
retrieved_docs2 = [doc.page_content for doc in docs2]
print(f"Retrieved {len(retrieved_docs2)} documents")

# --- Evaluate ---
eval_scores2 = []
for i, doc in enumerate(retrieved_docs2, 1):
    score = evaluator_chain.invoke({"query": query2, "document": doc})["relevance_score"]
    eval_scores2.append(float(score))
    print(f"  Doc {i}: relevance = {score:.2f}")

max_score2 = max(eval_scores2)
print(f"\nHighest score: {max_score2:.2f}")

# --- Decide action ---
sources2 = []

if max_score2 > 0.7:
    print("\nAction: CORRECT — Using retrieved document")
    best_doc2 = retrieved_docs2[eval_scores2.index(max_score2)]
    final_knowledge2 = best_doc2
    sources2.append(("Retrieved document", ""))

elif max_score2 < 0.3:
    print("\nAction: INCORRECT — Performing web search")
    rewritten_query2 = rewriter_chain.invoke({"query": query2})["query"].strip()
    print(f"Rewritten query: {rewritten_query2}")

    web_results2 = search.run(rewritten_query2)
    key_points2 = refiner_chain.invoke({"document": web_results2})["key_points"]
    final_knowledge2 = key_points2

    try:
        parsed2 = json.loads(web_results2)
        sources2 = [(r.get("title", "Untitled"), r.get("link", "")) for r in parsed2]
    except json.JSONDecodeError:
        sources2 = [("Web search", "")]

else:
    print("\nAction: AMBIGUOUS — Combining retrieved document + web search")
    best_doc2 = retrieved_docs2[eval_scores2.index(max_score2)]
    retrieved_kp2 = refiner_chain.invoke({"document": best_doc2})["key_points"]
    rewritten_query2 = rewriter_chain.invoke({"query": query2})["query"].strip()
    web_results2 = search.run(rewritten_query2)
    web_kp2 = refiner_chain.invoke({"document": web_results2})["key_points"]
    final_knowledge2 = retrieved_kp2 + "\n" + web_kp2
    sources2 = [("Retrieved document", ""), ("Web search", "")]

print(f"\nFinal knowledge preview: {str(final_knowledge2)[:300]}...")

Query: how did harry beat quirrell?

Retrieved 3 documents
  Doc 1: relevance = 0.00
  Doc 2: relevance = 0.00
  Doc 3: relevance = 0.00

Highest score: 0.00

Action: INCORRECT — Performing web search
Rewritten query: Harry Potter Quirrell defeat

Final knowledge preview: **Voldemort's Strategy:** Voldemort initially used Professor Quirrell as a host, latching onto the back of his head to attempt to steal the Philosopher's Stone.

**First Attempt on Harry's Life:** This event marked Voldemort's second attempt on Harry Potter's life.

**Date of Skirmish:** The skirmis...


---
## Step 10: Generate the Answer from Web Knowledge

In [11]:
sources_text2 = "\n".join([f"{title}: {link}" if link else title for title, link in sources2])

answer2 = response_chain.invoke({
    "query": query2,
    "knowledge": final_knowledge2,
    "sources": sources_text2
}).content

print(f"Question: {query2}")
print(f"\nAnswer: {answer2}")

Question: how did harry beat quirrell?

Answer: Harry defeated Quirrell by physically touching him. This caused Quirrell to be burned and disintegrate. This happened because Voldemort was possessing Quirrell as a host, and Harry's touch, fueled by a powerful bond of love, proved too much for Voldemort to withstand.



Sources: Web search


---
## Summary

| Test | Query | Scores | Action | Source |
|---|---|---|---|---|
| 1 | Climate change causes | ~0.85–0.95 | **Correct** | Used retrieved document directly |
| 2 | Harry Potter | ~0.0 | **Incorrect** | Fell back to web search |

**Key insight:** CRAG never generates from irrelevant context. When local documents don't match, it *corrects* by going to the web. This makes it more robust than standard RAG, which would generate from whatever it retrieved — relevant or not.